# Elo-Memory Quickstart

Bio-inspired episodic memory for AI agents. This notebook walks through the core workflow:

1. **Store** memories with automatic surprise scoring
2. **Retrieve** by similarity, location, entity, or time
3. **Consolidate** — replay and extract schemas
4. **Observe forgetting** — activation decay over time

```bash
pip install -e ".[dev]"  # from repo root
```

In [ ]:
import numpy as np
from datetime import datetime, timedelta
import tempfile, os

from elo_memory import (
    BayesianSurpriseEngine, SurpriseConfig,
    EpisodicMemoryStore, EpisodicMemoryConfig,
    TwoStageRetriever, RetrievalConfig,
    MemoryConsolidationEngine,
    ForgettingEngine,
)

tmpdir = tempfile.mkdtemp()
DIM = 64  # small for demo speed
np.random.seed(42)
print(f"Using temp dir: {tmpdir}")

## 1. Setup: Surprise Engine + Memory Store

In [ ]:
surprise_engine = BayesianSurpriseEngine(DIM)

store = EpisodicMemoryStore(
    EpisodicMemoryConfig(
        max_episodes=1000,
        embedding_dim=DIM,
        persistence_path=os.path.join(tmpdir, "demo_store"),
        consolidation_min_episodes=500,  # don't auto-consolidate during demo
    )
)
print(f"Store ready. Episodes: {len(store.episodes)}")

## 2. Store Memories with Surprise Scoring

Each memory gets a **surprise score** — how unexpected it is given what the system has seen before. High surprise = worth remembering.

In [ ]:
# Simulate a day of experiences
scenarios = [
    {"location": "office",  "entities": ["alice", "bob"],   "label": "morning standup"},
    {"location": "office",  "entities": ["alice"],          "label": "code review"},
    {"location": "cafe",   "entities": ["charlie"],         "label": "lunch with charlie"},
    {"location": "office",  "entities": ["alice", "bob"],   "label": "afternoon standup"},
    {"location": "office",  "entities": ["diana"],           "label": "1:1 with diana"},
    {"location": "park",   "entities": ["self"],             "label": "unexpected walk in park"},  # novel!
    {"location": "office",  "entities": ["alice", "bob"],   "label": "sprint planning"},
    {"location": "home",   "entities": ["self"],             "label": "evening reading"},
]

base_time = datetime.now() - timedelta(hours=8)
episodes = []

for i, scenario in enumerate(scenarios):
    # Generate embedding (in production, use SentenceTransformer)
    emb = np.random.randn(DIM).astype(np.float32)
    emb = emb / np.linalg.norm(emb)
    
    # Compute surprise
    surprise_info = surprise_engine.compute_surprise(emb)
    
    # Store
    ep = store.store_episode(
        content={"text": scenario["label"], "metadata": {}},
        embedding=emb,
        surprise=float(surprise_info["surprise"]),
        timestamp=base_time + timedelta(hours=i),
        location=scenario["location"],
        entities=scenario["entities"],
    )
    episodes.append(ep)
    print(f"  {scenario['label']:30s}  surprise={surprise_info['surprise']:6.2f}  novel={surprise_info['is_novel']}")

print(f"\nStored {len(store.episodes)} episodes")

## 3. Retrieve Memories

### By similarity (vector search via ChromaDB)

In [ ]:
# Query: something similar to the first episode
query_emb = episodes[0].embedding + np.random.randn(DIM).astype(np.float32) * 0.1
query_emb = query_emb / np.linalg.norm(query_emb)

results = store.retrieve_by_similarity(query_emb, k=3)
print("Top 3 by similarity:")
for ep in results:
    print(f"  {ep.content.get('text', '?'):30s}  importance={ep.importance:.3f}  location={ep.location}")

### By location and entity

In [ ]:
office_eps = store.retrieve_by_location("office")
print(f"Office episodes: {len(office_eps)}")
for ep in office_eps:
    print(f"  {ep.content.get('text', '?')}")

print()
alice_eps = store.retrieve_by_entity("alice")
print(f"Episodes with alice: {len(alice_eps)}")
for ep in alice_eps:
    print(f"  {ep.content.get('text', '?')}")

### Two-stage retrieval (similarity + temporal expansion)

In [ ]:
retriever = TwoStageRetriever(
    store,
    RetrievalConfig(
        similarity_threshold=0.0,  # low threshold for demo
        max_retrieved=5,
        similarity_weight=0.5,
        recency_weight=0.3,
        importance_weight=0.2,
    ),
)

results = retriever.retrieve(query=query_emb)
print(f"Two-stage results ({len(results)} episodes):")
for ep, score in results:
    print(f"  score={score:.3f}  {ep.content.get('text', '?'):30s}  location={ep.location}")

## 4. Consolidation — Replay and Schema Extraction

Mimics sleep-dependent memory consolidation: replay high-priority episodes and extract patterns (schemas).

In [ ]:
consolidation = MemoryConsolidationEngine()
stats = consolidation.consolidate(store.episodes)

print("Consolidation stats:")
for k, v in stats.items():
    print(f"  {k}: {v}")

schemas = consolidation.get_schema_summary()
print(f"\nExtracted {len(schemas)} schemas:")
for s in schemas:
    print(f"  {s['pattern']}  (freq={s['frequency']}, entities={s['common_entities']})")

## 5. Forgetting — Activation Decay Over Time

Memories decay following the Ebbinghaus forgetting curve. Rehearsal (retrieval) boosts activation.

In [ ]:
forgetting = ForgettingEngine()

# Track activation over time for one episode
ep = episodes[0]
print(f"Episode: {ep.content.get('text', '?')}")
print(f"Initial importance: {ep.importance:.3f}\n")

print(f"{'Hours':>8s}  {'No rehearsal':>14s}  {'3 rehearsals':>14s}")
print("-" * 42)
for hours in [0, 1, 6, 12, 24, 48, 168]:
    t = ep.timestamp + timedelta(hours=hours)
    a0 = forgetting.compute_activation(ep.importance, ep.timestamp, rehearsal_count=0, current_time=t)
    a3 = forgetting.compute_activation(ep.importance, ep.timestamp, rehearsal_count=3, current_time=t)
    print(f"{hours:8d}  {a0:14.4f}  {a3:14.4f}")

## 6. Persistence — Save and Load

In [ ]:
# Save
store.save_state()
print(f"Saved {len(store.episodes)} episodes to disk")

# Load into a fresh store
store2 = EpisodicMemoryStore(
    EpisodicMemoryConfig(
        embedding_dim=DIM,
        persistence_path=os.path.join(tmpdir, "demo_store"),
        consolidation_min_episodes=500,
    )
)
store2.load_state()
print(f"Loaded {len(store2.episodes)} episodes from disk")

# Verify round-trip
assert len(store2.episodes) == len(store.episodes)
print("Round-trip OK")

## 7. Statistics

In [ ]:
stats = store.get_statistics()
for k, v in stats.items():
    print(f"  {k}: {v}")